# 03 — Demo interativa: filtrando ECG ao vivo

Escolha o registro, o tipo de ruído, o SNR e o filtro; o painel mostra o sinal no tempo, o espectro (FFT) e as métricas em tempo real. (Requer `ipywidgets`.)

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # raiz do repo no path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
import ipywidgets as widgets
from IPython.display import display
from src.data_io import ensure_datasets, load_ecg, DEFAULT_RECORDS, FS
from src.noise.contaminate import contaminate
from src.filters.notch import apply_notch
from src.filters.highpass import apply_highpass
from src.filters.lowpass_fir import apply_lowpass_fir
from src.metrics.metrics import summary
from src.viz import plots
ensure_datasets()

In [3]:
FILTROS = {
    'Notch 60Hz': lambda y: apply_notch(y),
    'Passa-alta 0.5Hz': lambda y: apply_highpass(y),
    'Passa-baixa FIR 40Hz': lambda y: apply_lowpass_fir(y, zero_phase=True),
}
_cache = {}
def _load(rec):
    if rec not in _cache: _cache[rec] = load_ecg(rec)
    return _cache[rec]

def atualizar(record, ruido, snr, filtro):
    clean, fs = _load(record)
    ref = apply_highpass(clean) if ruido == 'bw' else clean
    noisy = contaminate(clean, ruido, snr_db=snr)
    noisy_ref = noisy - clean + ref
    filt = FILTROS[filtro](noisy)
    fig, axs = plt.subplots(1, 2, figsize=(13,4))
    plots.plot_time_overlay(clean=ref, noisy=noisy_ref, filtered=filt, fs=fs, n=3*fs, ax=axs[0])
    plots.plot_spectrum(noisy_ref[:6*fs], fs=fs, ax=axs[1], label='ruidoso')
    plots.plot_spectrum(filt[:6*fs], fs=fs, ax=axs[1], label='filtrado')
    m = summary(ref, noisy_ref, filt)
    print(f"SNR {m['snr_in_db']:+.1f} -> {m['snr_out_db']:+.1f} dB (Δ{m['snr_improvement_db']:+.1f}) | "
          f"PRD {m['prd']:.1f}% | corr {m['corr']:.3f}")
    plt.show()

In [4]:
widgets.interact(
    atualizar,
    record=widgets.Dropdown(options=list(DEFAULT_RECORDS), value='100', description='registro'),
    ruido=widgets.Dropdown(options=['60hz','bw','ma','em'], value='60hz', description='ruído'),
    snr=widgets.IntSlider(min=0, max=24, step=2, value=6, description='SNR dB'),
    filtro=widgets.Dropdown(options=list(FILTROS), value='Notch 60Hz', description='filtro'),
);

interactive(children=(Dropdown(description='registro', options=('100', '103', '105', '115', '215'), value='100…